# Enclave Evaluation — Model Owner

| Actor | Email | Role |
|-------|-------|------|
| **Enclave** | `enclave@ai-safety.org` | Trusted execution environment (sealed memory) |
| **Model Owner** | (this notebook) | Owns the private language-model weights (Qwen) |
| **Benchmark Owner** | `benchmark_owner@ai-safety.org` | Owns the private Vietnamese-harm benchmark |
| **Researcher** | `researcher@ai-safety.org` | Writes the eval code, submits the job |

This notebook drives only the **Model Owner** steps; the Benchmark Owner and Researcher run their own notebooks in parallel.

## Setup

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path

os.environ["PRE_SYNC"] = "false"

from syft_enclaves import login_do

# Repo root — so models/ and code/ resolve whether the kernel starts at the repo
# root or in notebooks/nbsplit/.
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
print(f"  repo root : {ROOT}")

In [ ]:
ENCLAVE_EMAIL     = "enclave@ai-safety.org"
MODEL_OWNER_EMAIL = "model_owner@ai-safety.org"
RESEARCHER_EMAIL  = "researcher@ai-safety.org"

print(f"  Enclave: {ENCLAVE_EMAIL}  |  Researcher: {RESEARCHER_EMAIL}")

## Step 0 — Log in as Model Owner

In [ ]:
model_owner = login_do(MODEL_OWNER_EMAIL)
print(f"  Model Owner : {model_owner.email}")

In [ ]:
# Optionally clear state for a fresh run
model_owner._manager.delete_syftbox()
model_owner._manager.peer_manager.write_own_version()

### Launch the enclave

Start the enclave service for `enclave@ai-safety.org` out-of-band. It runs the job
automatically once **both** owners approve — no notebook drives it.

## Step 1 — Peer with the Enclave

In [ ]:
model_owner.add_peer(ENCLAVE_EMAIL)
model_owner.sync()
print(f"  Model Owner peered with enclave ({ENCLAVE_EMAIL})")

### Step 1.1 — Wait for the Researcher peer request, then approve

The Researcher notebook adds you as a peer. Re-run the cell below until their request appears, then approve.

In [ ]:
model_owner.sync()
model_owner.peers

In [ ]:
model_owner.approve_peer_request(RESEARCHER_EMAIL, peer_must_exist=False)
model_owner.sync()
print("  Researcher peer approved")

### Step 1.2 — Attest enclave's identity

In [ ]:
# Wait for the enclave to accept the peer request, then attest
model_owner.attest_peer(ENCLAVE_EMAIL)

## Step 2.1 — Model assets

The model is set once in `MODEL` (the lightest is `qwen2.5-0.5b`). Download weights first:
`uv run python scripts/download_models.py --models qwen2.5-0.5b`

In [ ]:
MODEL = "qwen2.5-0.5b"  # the model under audit — single source of truth: the weights dir
MODEL_DIR = ROOT / "models" / MODEL
assert MODEL_DIR.exists(), (
    f"missing {MODEL_DIR} — run: uv run python scripts/download_models.py --models {MODEL}"
)


def mock_model() -> Path:
    # Public mock: a stub infer.py with the same init/generate interface as the private
    # one (no weights), so the researcher can develop their eval before any real weights.
    d = Path(tempfile.mkdtemp()) / "model-mock"
    d.mkdir(parents=True, exist_ok=True)
    shutil.copy(ROOT / "code" / "model_owner_code" / "mock_infer.py", d / "infer.py")
    return d

## Step 2.2 — Upload the model weights

* **mock**: a stub `infer.py` (no weights) — the researcher develops against this
* **private**: real Qwen weights + the model owner's `infer.py` — only the enclave receives this

In [ ]:
# Private asset = real weights + the model owner's real infer.py (their IP).
shutil.copy(ROOT / "code" / "model_owner_code" / "infer.py", MODEL_DIR / "infer.py")
mock_dir = mock_model()
_gb = sum(p.stat().st_size for p in MODEL_DIR.rglob("*") if p.is_file()) / 1e9

model_owner.create_dataset(
    name="model",
    mock_path=mock_dir,
    private_path=str(MODEL_DIR),
    summary="Qwen model weights (mlx safetensors) + infer.py — private to the enclave",
    users=[RESEARCHER_EMAIL, ENCLAVE_EMAIL],
    upload_private=True,
    sync=False,
)
model_owner.share_private_dataset("model", ENCLAVE_EMAIL)
model_owner.sync()
print(f"  Model Owner uploaded 'model'  (private: {_gb:.2f} GB)")

## Step 3 — Wait for the Researcher to submit the job, then approve

The Researcher submits `vn_harm_audit` to the enclave. Re-sync until it appears, inspect the
submitted code, then approve. The enclave only runs once **both** owners approve.

In [ ]:
JOB_NAME = "vn_harm_audit"
model_owner.sync()
model_owner_job = model_owner.jobs[JOB_NAME]
print(f"  Model Owner sees '{JOB_NAME}'  status={model_owner_job.status}")
model_owner_job  # inspect the submitted code before approving

In [ ]:
model_owner.approve_job(model_owner_job)
model_owner.sync()
print("  Model Owner approved")

## Step 4 — Receive results

Results arrive here because the Researcher submitted with `share_results_with_do=True`.

In [ ]:
model_owner.sync()
model_owner_job = model_owner.jobs[JOB_NAME]
print(f"  Output files : {model_owner_job.output_paths}")
assert len(model_owner_job.output_paths) > 0